In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 1. โหลดข้อมูลจริงของคุณ
# แทนที่ด้วยไฟล์จริงของคุณ เช่น df = pd.read_csv('your_data.csv')
df = pd.read_csv(r"C:\Users\msapi\OneDrive\Documents\Network Anomaly Dataset\networkanomalydataset.csv")

# 2. เตรียมข้อมูล (Preprocessing)
# เลือกเฉพาะคอลัมน์ตัวเลขที่ต้องการนำมาตรวจจับ Anomaly
# หรือระบุชื่อคอลัมน์ตรงๆ เช่น features = ['amount', 'usage_time', 'error_count']
features = df.select_dtypes(include=['number']).columns.tolist()

# จัดการ Missing Values (ตัวอย่าง: เติมด้วยค่าเฉลี่ย หรือจะดรอปทิ้ง df.dropna())
X = df[features].fillna(df[features].median())

# (ทางเลือก) ปรับสเกลข้อมูล ถ้าข้อมูลแต่ละคอลัมน์มีหน่วยต่างกันมากๆ
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. สร้างและรัน Isolation Forest Model
iso_forest = IsolationForest(
    n_estimators=200,          # เพิ่มจำนวน tree ให้ผลลัพธ์นิ่งขึ้น
    contamination='auto',      # ใช้ 'auto' ถ้าไม่รู้สัดส่วน หรือใส่ เช่น 0.02 (คาดว่ามี 2%)
    max_samples='auto',
    random_state=42,
    n_jobs=-1                  # ใช้ CPU ทุกคอร์เพื่อความเร็ว
)

# 4. บันทึกผลลัพธ์กลับเข้า DataFrame เดิม
# anomaly_label: 1 คือปกติ, -1 คือผิดปกติ
df['anomaly_label'] = iso_forest.fit_predict(X_scaled)

# anomaly_score: ค่ายิ่งติดลบมาก = ยิ่งผิดปกติมาก
df['anomaly_score'] = iso_forest.decision_function(X_scaled)

# 5. กรองและบันทึกข้อมูลที่ผิดปกติ
# ดึงเฉพาะรายการที่โมเดลระบุว่าเป็น anomaly (-1) เรียงจากที่น่าสงสัยที่สุด
anomalies = df[df['anomaly_label'] == 1].sort_values(by='anomaly_score')

print(f"ตรวจพบข้อมูลผิดปกติทั้งหมด: {len(anomalies)} รายการ (จากทั้งหมด {len(df)} รายการ)")
print("\nตัวอย่าง 5 รายการที่ผิดปกติมากที่สุด:")
print(anomalies[features + ['anomaly_score']].head())

# บันทึกผลลัพธ์ออกเป็นไฟล์ใหม่เพื่อนำไปตรวจสอบต่อ
anomalies.to_csv('detected_anomalies.csv', index=False)

ตรวจพบข้อมูลผิดปกติทั้งหมด: 937 รายการ (จากทั้งหมด 1654 รายการ)

ตัวอย่าง 5 รายการที่ผิดปกติมากที่สุด:
      Inbound Rate(bit/s)  Outbound Rate(bit/s)  \
1436            -0.794574             -0.773147   
1097            -0.854135             -0.838965   
1588            -0.795965             -0.771914   
892             -0.806081             -0.832389   
740              1.378660              1.239901   

      Inbound Bandwidth Utilization(%)  Outbound Bandwidth Utilization(%)  \
1436                         -0.794587                          -0.772687   
1097                         -0.854043                          -0.839492   
1588                         -0.796072                          -0.772449   
892                          -0.805985                          -0.832478   
740                           1.378443                           1.239254   

      Label  anomaly_score  
1436      1       0.000206  
1097      1       0.000394  
1588      1       0.001139  
892       1